# Accessibility
**Accessibility** represents the average accumulated travel cost required to reach each raster cell from regularly spaced starting points within the study area. 
## Input
- Digital Elevation Model (DEM)
- Vertical factor table (*inTable*)
- Starting points (*Starting_points*)
- Barrier (*Barrier*)
- Cost raster (*Cost_Raster*)
- Aspect raster (*Aspect*)
## Method
tarting points were generated using the *Fishnet* tool at a regular spacing of 1000 m.
For each starting point:
- An anisotropic distance accumulation was calculated using:
- A slope-dependent vertical factor,
- A vegetation-based cost surface,
- Lake areas defined as barriers,
- A maximum slope-dependent cart travel distance of 8000.
Each resulting distance accumulation raster was stored in *TotalCost.gdb*. Additionally, a binary reachability raster was created for each iteration, indicating whether a cell was reached (1) or not reached (0).
Finally, all distance accumulation rasters were aggregated using cell-wise summation. The summed cost raster was divided by the summed reachability raster. The resulting raster therefore represents, for each cell, the mean accumulated travel cost from all starting points from which that cell is reachable.

## Implementation instructions
- Add all data under *data/01_Covariate/01.3_Accessibility/Accessibility.gdb* to the current map
- Indicate the link to the vertical factor (*cart.csv*) in the script
- Adjust input manually (number of iterations, Distance Accumulation)
- Execute the cell

In [ ]:
# Accessibility Model
import arcpy
from arcpy.sa import *
import os

# ----------------------- Configuration -----------------------
# set parallel cores
arcpy.env.parallelProcessingFactor = "100%"

# define digital elevation model
in_raster = r"DEM"

# configure environment settings
arcpy.env.extent = in_raster
arcpy.env.cellSize = in_raster
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
arcpy.env.workspace = "memory"

# create describe object
desc = arcpy.Describe(arcpy.Describe(in_raster).path)
print(desc.path)

# create new file geodatabase
arcpy.management.CreateFileGDB(desc.path,"TotalCost.gdb")
arcpy.management.CreateFileGDB(desc.path,"TotalCost_null.gdb")
out_path = os.path.join(desc.path,"TotalCost.gdb")
out_path_null = os.path.join(desc.path,"TotalCost_null.gdb")
print(out_path)
print(out_path_null)

# ----------------------- MODELLING -----------------------
# set vertical factor
inTable = "link_to_cart/Cart.txt" #<-- link to the cart.csv
myVerticalFactor = VfTable(inTable)

#create loop
fc = "Starting_points"
fields = ['OID@', 'SHAPE@']
n = arcpy.management.GetCount(fc)
with arcpy.da.SearchCursor(fc, fields) as cursor:
    for row in cursor:
        #create variable of geometry
        point = row[1]
        
        #create variable for name
        name = "distance"+str(row[0]).zfill(len(str(n)))
                
        #Cost Distance
        distance = DistanceAccumulation(point, "Barrier", "DEM", "Cost_Raster", "DEM",myVerticalFactor,"Aspect", "LINEAR 0.5 181 1.11111111111111E-02", "","","","0","8000", "", "", "PLANAR")
        out_raster = os.path.join(out_path,name)
        distance.save(out_raster)
        
        #create variable for null name
        nullname = "null"+str(row[0]).zfill(len(str(n)))
        
        #null raster
        outIsNull = IsNull(distance)
        outIsNull_invert = (outIsNull-1)*(-1)
        out_raster_null = os.path.join(out_path_null,nullname)
        outIsNull_invert.save(out_raster_null)
        
        print('cost distance {} of {}'.format(str(row[0]), n))
        
# ----------------------- Aggregation -----------------------
print('')
print("Agregating results...")

# set a new environment
arcpy.env.workspace = out_path

# create a list of all remaining rasters
print('')
print("calculating cost raster")
in_rasters = arcpy.ListRasters("*", "All")
fcCount = len(in_rasters)
print(fcCount)
localsum = arcpy.sa.CellStatistics(in_rasters,"SUM","DATA")
name = "localsum"
out_raster = os.path.join(out_path,name)
localsum.save(out_raster)
print("calculating null raster")
arcpy.env.workspace = out_path_null
in_rasters = arcpy.ListRasters("*", "All")
fcCount = len(in_rasters)
print(fcCount)
localsum_null = arcpy.sa.CellStatistics(in_rasters,"SUM","DATA")
name_null = "localsum_null"
out_raster_null = os.path.join(out_path_null,name_null)
localsum_null.save(out_raster_null)

# aggregate all rasters
mean = Divide(localsum, localsum_null)
out_result = os.path.join(out_path,"Accessibility")
mean.save(out_result)
print("completed")